In [1]:
import json
from pathlib import Path

import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
from tqdm import tqdm

In [2]:
db_path = "../data/final/lancedb"
table_name = "document_chunks"
chunk_dir = "../data/interim/docling"
embedding_model = "distiluse-base-multilingual-cased-v1"

In [3]:
def read_all_chunks(data_dir: str) -> list[dict[str, str]]:
    """
    Read all chunks from JSON files and return as a flat list of dictionaries.

    Args:
        data_dir: Base directory containing the docling subdirectories

    Returns:
        List of dictionaries with 'document' and 'text' keys
    """
    result = []
    base_path = Path(data_dir)

    if not base_path.exists():
        raise Exception("directory not found")

    # Find all chunks.json files in subdirectories
    chunks_files = base_path.glob("*/chunks.json")

    for chunks_file in chunks_files:
        # Extract the document name (the * part from data/interim/docling/*/chunks.json)
        document_name = chunks_file.parent.name

        # Read the JSON file
        try:
            with chunks_file.open("r", encoding="utf-8") as f:
                chunks_data = json.load(f)

            # Handle both cases: if chunks_data is a list of strings
            if isinstance(chunks_data, list):
                for chunk_text in chunks_data:
                    if isinstance(chunk_text, str):
                        result.append({"document": document_name, "text": chunk_text})
        except Exception as e:
            print(f"Error reading {chunks_file}: {e}")

    return result

In [4]:
chunks = read_all_chunks(chunk_dir)

In [5]:
embedder = (
    get_registry()
    .get("sentence-transformers")
    .create(name="distiluse-base-multilingual-cased-v1")
)
db = lancedb.connect(db_path)

In [6]:
class Schema(LanceModel):
    text: str = embedder.SourceField()
    vector: Vector(embedder.ndims()) = embedder.VectorField()
    document: str

In [7]:
reranker = ColbertReranker()

Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device mps
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


In [9]:
table = db.create_table(table_name, schema=Schema, mode="overwrite")

In [ ]:
batch_size = 1000
total_batches = (len(chunks) + batch_size - 1) // batch_size

for i in tqdm(range(0, len(chunks), batch_size), total=total_batches):
    batch = chunks[i : i + batch_size]
    table.add(batch)

Adding batches:  18%|██████████▏                                               | 7/40 [00:56<04:16,  7.78s/it]

In [ ]:
table.create_fts_index("text", replace=True)
table.wait_for_index(["text_idx"])

In [ ]:
query = "добрый день подскажите мою карту украли как мне заблокировать карту"

res = (
    table.search(
        query,
        query_type="hybrid",
        vector_column_name="vector",
        fts_columns="text",
    )
    .limit(50)
    .rerank(reranker)
    .to_list()
)

for r in res:
    r = dict(r)
    del r["vector"]
    print(r)